In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

results_dir = Path("../results/split_ratios")
scalars = TBScalars(".cache/split_ratios")

In [ ]:
res_df = []
for test in tqdm([*results_dir.iterdir()]):
    env, wm_ratio, rl_ratio, seed = test.name.split("-")
    wm_ratio = int(wm_ratio.removeprefix("wm_ratio="))
    rl_ratio = int(rl_ratio.removeprefix("rl_ratio="))
    seed = int(seed.removeprefix("seed="))
    res_df.append(
        {
            "path": test,
            "env": env,
            "wm_ratio": wm_ratio,
            "rl_ratio": rl_ratio,
            "seed": seed,
        }
    )
    scalars.read(test)
res_df = pd.DataFrame.from_records(res_df)
res_df

In [ ]:
wm_ratios = res_df["wm_ratio"].unique()
rl_ratios = res_df["rl_ratio"].unique()

fig = make_subplots(
    rows=len(wm_ratios),
    row_titles=[str(x) for x in wm_ratios],
    cols=len(rl_ratios),
    column_titles=[str(x) for x in rl_ratios],
)

color = next(make_color_iter())
for row, wm_ratio in enumerate(wm_ratios, 1):
    for col, rl_ratio in enumerate(rl_ratios, 1):
        res_dfs = res_df[
            (res_df["wm_ratio"] == wm_ratio) & (res_df["rl_ratio"] == rl_ratio)
        ]
        dfs = []
        for _, test in res_dfs.iterrows():
            df = scalars.read(test["path"])
            df = df[df["tag"] == "val/mean_ep_ret"]
            df["index"] = np.arange(len(df))
            dfs.append(df)
        df = pd.concat(dfs)

        g = df.groupby("index")
        avg_df = pd.DataFrame.from_records(
            {
                "score_mean": g["value"].mean(),
                "score_std": g["value"].std(),
                "step": g["step"].median(),
            }
        )

        for trace in err_line(
            x=avg_df["step"],
            y=avg_df["score_mean"],
            std=avg_df["score_std"],
            color=color,
        ):
            fig.add_trace(trace, row=row, col=col)

fig

In [ ]:
final_scores = []
for _, test in res_df.iterrows():
    df = scalars.read(test["path"])
    df = df[df["tag"] == "val/mean_ep_ret"]
    last = df.iloc[-1]["value"]
    final_scores.append({"path": test["path"], "score": last})
final_scores = pd.DataFrame.from_records(final_scores)
final_scores = pd.merge(final_scores, res_df, on="path")
final_scores

In [ ]:
px.scatter(final_scores, x="wm_ratio", y="score", log_x=True)

In [ ]:
px.scatter(final_scores, x="rl_ratio", y="score", log_x=True)